# Landslide combined-class step 06: expected annual damage maps (minimum_scenario)

Runs the landslide EAD map workflow against combined-class EAD outputs from step 05.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

try:
    from IPython.display import display
except Exception:
    def display(display_value):
        print(display_value)


In [ ]:
# Core paths
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
results_path = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_minimum_scenario_combined_class'
output_damage_estimates = results_path / 'damage_estimates'

asset_ead_file = output_damage_estimates / 'landslide_ead_asset_level_usd_combined_class.csv'
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

data_root = base_path / 'dphil_common_cross_cutting/common_incoming_data'

for required in [asset_ead_file, network_csv, jamaica_boundary_path]:
    if not required.exists():
        raise FileNotFoundError(f'Missing required file: {required}')

print('Asset EAD file:', asset_ead_file)
print('Network csv:', network_csv)
print('Jamaica boundary:', jamaica_boundary_path)


In [ ]:
# Load EAD output and compute avoided metrics of interest
asset_ead = pd.read_csv(asset_ead_file)
required_cols = [
    'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
    'EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD'
]
missing_cols = [required_column for required_column in required_cols if required_column not in asset_ead.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in asset EAD file: {missing_cols}')

asset_ead = asset_ead.copy()
asset_ead['Avoided_EAD_Reafforestation_USD'] = asset_ead['EAD_Baseline_USD'] - asset_ead['EAD_Reafforestation_USD']
asset_ead['Avoided_EAD_Protection_USD'] = asset_ead['EAD_Deforestation_USD'] - asset_ead['EAD_Baseline_USD']
asset_ead['Combined_Benefit_Reafforestation_vs_Deforestation_USD'] = asset_ead['EAD_Deforestation_USD'] - asset_ead['EAD_Reafforestation_USD']

print(f'Loaded asset EAD rows: {len(asset_ead):,}')
display(asset_ead.head(10))


In [ ]:
# Build geospatial asset map layer by joining asset geometries with asset-level EAD metrics
network_details = pd.read_csv(network_csv)
network_details = network_details[[
    'sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column', 'path'
]].drop_duplicates().copy()


def resolve_network_asset_file(asset_relative_path):
    relative_asset_path = Path(asset_relative_path)
    asset_file_in_common_incoming_data = data_root / relative_asset_path
    asset_file_in_nested_networks_folder = data_root / 'networks' / relative_asset_path

    if asset_file_in_common_incoming_data.exists():
        return asset_file_in_common_incoming_data
    if asset_file_in_nested_networks_folder.exists():
        return asset_file_in_nested_networks_folder

    raise FileNotFoundError(
        f"Could not find asset file '{relative_asset_path}'. Checked: {asset_file_in_common_incoming_data} ; {asset_file_in_nested_networks_folder}"
    )


include_zero_assets = False
map_layers = []
missing_asset_files = []

for row in network_details.itertuples(index=False):
    ead_subset = asset_ead.loc[
        (asset_ead['Asset'] == row.asset_gpkg) & (asset_ead['Layer'] == row.asset_layer),
        [
            'Asset_ID',
            'EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD',
            'Avoided_EAD_Reafforestation_USD', 'Avoided_EAD_Protection_USD',
            'Combined_Benefit_Reafforestation_vs_Deforestation_USD'
        ]
    ].copy()

    if ead_subset.empty:
        continue

    if not include_zero_assets:
        ead_subset = ead_subset[
            (ead_subset['Avoided_EAD_Reafforestation_USD'] != 0)
            | (ead_subset['Avoided_EAD_Protection_USD'] != 0)
        ].copy()

    if ead_subset.empty:
        continue

    try:
        asset_file = resolve_network_asset_file(row.path)
    except FileNotFoundError:
        missing_asset_files.append(row.path)
        continue

    asset_gdf = gpd.read_file(asset_file, layer=row.asset_layer)
    if row.asset_id_column not in asset_gdf.columns:
        print(f"Skipping {row.asset_gpkg}_{row.asset_layer}: missing id column '{row.asset_id_column}'")
        continue

    asset_gdf = asset_gdf[[row.asset_id_column, 'geometry']].copy()
    asset_gdf = gpd.GeoDataFrame(asset_gdf, geometry='geometry', crs=asset_gdf.crs)
    if asset_gdf.crs is not None:
        asset_gdf = asset_gdf.to_crs('EPSG:3448')

    asset_gdf['_join_id'] = asset_gdf[row.asset_id_column].astype(str)
    ead_subset['_join_id'] = ead_subset['Asset_ID'].astype(str)

    merged = asset_gdf.merge(ead_subset, on='_join_id', how='inner')
    if merged.empty:
        continue

    merged['Sector'] = row.sector
    merged['Subsector'] = row.asset_description
    merged['Asset'] = row.asset_gpkg
    merged['Layer'] = row.asset_layer

    map_layers.append(merged[[
        'Sector', 'Subsector', 'Asset', 'Layer', row.asset_id_column,
        'Asset_ID',
        'EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD',
        'Avoided_EAD_Reafforestation_USD', 'Avoided_EAD_Protection_USD',
        'Combined_Benefit_Reafforestation_vs_Deforestation_USD',
        'geometry'
    ]].rename(columns={row.asset_id_column: 'Asset_ID_Source'}))

if not map_layers:
    raise ValueError('No map features were built. Check step-05 outputs and asset files.')

landslide_ead_map_layers = gpd.GeoDataFrame(
    pd.concat(map_layers, ignore_index=True),
    geometry='geometry',
    crs='EPSG:3448'
)

map_layers_file = output_damage_estimates / 'landslide_source_and_runout_ead_asset_map_layers_combined_class.gpkg'
landslide_ead_map_layers.to_file(map_layers_file, driver='GPKG')

print(f'Map features loaded: {len(landslide_ead_map_layers):,}')
print('Features by sector:')
display(landslide_ead_map_layers.groupby('Sector', as_index=False).size())
print('Saved map layer:', map_layers_file)

if missing_asset_files:
    print('Missing asset files skipped:')
    for missing_asset_path in sorted(set(missing_asset_files)):
        print('-', missing_asset_path)


In [ ]:
# Overall and sector totals for avoided EAD metrics (USD + baseline percentage statistics)

def format_usd_readable(value):
    if pd.isna(value):
        return 'NA'
    abs_value = abs(float(value))
    sign = '-' if float(value) < 0 else ''
    if abs_value >= 1_000_000_000:
        return f"{sign}${abs_value / 1_000_000_000:,.2f} billion"
    if abs_value >= 1_000_000:
        return f"{sign}${abs_value / 1_000_000:,.2f} million"
    if abs_value >= 1_000:
        return f"{sign}${abs_value / 1_000:,.2f} thousand"
    return f"{sign}${abs_value:,.2f}"


def format_pct(value):
    if pd.isna(value):
        return 'NA'
    return f"{float(value):,.2f}%"


def add_avoided_ead_baseline_stats(summary_df):
    summary_df = summary_df.copy()
    baseline_ead = summary_df['EAD_Baseline_USD'].replace({0: np.nan})
    reafforestation_avoided = summary_df['Avoided_EAD_Reafforestation_USD']
    protection_avoided = summary_df['Avoided_EAD_Protection_USD']
    combined_benefit = summary_df['Combined_Benefit_Reafforestation_vs_Deforestation_USD']

    summary_df['Avoided_EAD_Reafforestation_Pct_of_Baseline_EAD'] = 100.0 * reafforestation_avoided / baseline_ead
    summary_df['Avoided_EAD_Protection_Pct_of_Baseline_EAD'] = 100.0 * protection_avoided / baseline_ead
    summary_df['Combined_Benefit_Pct_of_Baseline_EAD'] = 100.0 * combined_benefit / baseline_ead
    summary_df['Protection_vs_Reafforestation_Avoided_EAD_Difference_USD'] = protection_avoided - reafforestation_avoided

    reafforestation_avoided_denom = reafforestation_avoided.replace({0: np.nan})
    summary_df['Protection_vs_Reafforestation_Avoided_EAD_Difference_Pct_of_Reafforestation_Avoided_EAD'] = (
        100.0 * summary_df['Protection_vs_Reafforestation_Avoided_EAD_Difference_USD'] / reafforestation_avoided_denom
    )

    readable_cols = [
        'EAD_Baseline_USD',
        'EAD_Deforestation_USD',
        'EAD_Reafforestation_USD',
        'Avoided_EAD_Reafforestation_USD',
        'Avoided_EAD_Protection_USD',
        'Combined_Benefit_Reafforestation_vs_Deforestation_USD',
        'Protection_vs_Reafforestation_Avoided_EAD_Difference_USD',
    ]
    for col in readable_cols:
        summary_df[f'{col}_Readable'] = summary_df[col].apply(format_usd_readable)

    pct_cols = [
        'Avoided_EAD_Reafforestation_Pct_of_Baseline_EAD',
        'Avoided_EAD_Protection_Pct_of_Baseline_EAD',
        'Combined_Benefit_Pct_of_Baseline_EAD',
        'Protection_vs_Reafforestation_Avoided_EAD_Difference_Pct_of_Reafforestation_Avoided_EAD',
    ]
    for col in pct_cols:
        summary_df[f'{col}_Label'] = summary_df[col].apply(format_pct)

    return summary_df


ead_cols = [
    'EAD_Baseline_USD',
    'EAD_Deforestation_USD',
    'EAD_Reafforestation_USD',
]
summary_cols = [
    'Avoided_EAD_Reafforestation_USD',
    'Avoided_EAD_Protection_USD',
    'Combined_Benefit_Reafforestation_vs_Deforestation_USD'
]
metric_cols = ead_cols + summary_cols

overall_summary = pd.DataFrame([asset_ead[metric_cols].sum()])
overall_summary = add_avoided_ead_baseline_stats(overall_summary)

sector_summary = asset_ead.groupby('Sector', as_index=False)[metric_cols].sum()
sector_summary = add_avoided_ead_baseline_stats(sector_summary)

overall_file = output_damage_estimates / 'landslide_ead_avoided_overall_totals_usd_combined_class.csv'
sector_file = output_damage_estimates / 'landslide_ead_avoided_sector_totals_usd_combined_class.csv'
overall_summary.to_csv(overall_file, index=False)
sector_summary.to_csv(sector_file, index=False)

print('Overall totals:')
display(overall_summary)
print('Sector totals:')
display(sector_summary.sort_values('Sector'))
print('Saved:', overall_file)
print('Saved:', sector_file)


In [ ]:
# Subsector breakdown table (within-sector drivers + baseline percentage statistics)
subsector_summary = (
    asset_ead
    .groupby(['Sector', 'Subsector'], as_index=False)[metric_cols]
    .sum()
)
subsector_summary = add_avoided_ead_baseline_stats(subsector_summary)

# Shares of sector totals by metric to show what drives each sector result
sector_totals_for_share = sector_summary.set_index('Sector')[summary_cols]

for metric_col, share_col in [
    ('Avoided_EAD_Reafforestation_USD', 'Reafforestation_Share_of_Sector'),
    ('Avoided_EAD_Protection_USD', 'Protection_Share_of_Sector'),
    ('Combined_Benefit_Reafforestation_vs_Deforestation_USD', 'Combined_Benefit_Share_of_Sector'),
]:
    subsector_summary[share_col] = np.nan
    for sector_name in subsector_summary['Sector'].unique():
        denom = float(sector_totals_for_share.loc[sector_name, metric_col]) if sector_name in sector_totals_for_share.index else 0.0
        mask = subsector_summary['Sector'] == sector_name
        if denom != 0:
            subsector_summary.loc[mask, share_col] = subsector_summary.loc[mask, metric_col] / denom
        else:
            subsector_summary.loc[mask, share_col] = pd.NA

subsector_summary['Reafforestation_Share_of_Sector_Pct'] = (100.0 * subsector_summary['Reafforestation_Share_of_Sector']).round(2)
subsector_summary['Protection_Share_of_Sector_Pct'] = (100.0 * subsector_summary['Protection_Share_of_Sector']).round(2)
subsector_summary['Combined_Benefit_Share_of_Sector_Pct'] = (100.0 * subsector_summary['Combined_Benefit_Share_of_Sector']).round(2)

subsector_file = output_damage_estimates / 'landslide_ead_avoided_subsector_totals_usd_combined_class.csv'
subsector_summary.to_csv(subsector_file, index=False)

print('Subsector totals (drivers within each sector):')
display(subsector_summary.sort_values(['Sector', 'Subsector']))
print('Saved:', subsector_file)


In [ ]:
# Grouped bar chart: sector and total EAD by scenario (USD million)
chart_sector_order = ['buildings', 'transport', 'water', 'energy']
scenario_columns = [
    ('EAD_Baseline_USD', 'Baseline'),
    ('EAD_Deforestation_USD', 'Deforestation'),
    ('EAD_Reafforestation_USD', 'Reafforestation'),
]
scenario_value_columns = [scenario_definition[0] for scenario_definition in scenario_columns]

sector_chart = (
    asset_ead
    .groupby('Sector', as_index=False)[scenario_value_columns]
    .sum()
)

# Ensure all expected sectors are present and in a stable order.
sector_chart = sector_chart.set_index('Sector')
for sector_name in chart_sector_order:
    if sector_name not in sector_chart.index:
        sector_chart.loc[sector_name, scenario_value_columns] = 0.0
sector_chart = sector_chart.loc[chart_sector_order]

total_row = pd.DataFrame(
    {
        scenario_column: [float(sector_chart[scenario_column].sum())]
        for scenario_column in scenario_value_columns
    },
    index=['TOTAL'],
)

sector_chart = pd.concat([sector_chart, total_row], axis=0)
sector_chart_musd = sector_chart / 1_000_000.0

x = np.arange(len(sector_chart_musd.index))
bar_width = 0.24

scenario_styles = {
    'Baseline': {'color': '#b3b3b3', 'hatch': ''},
    'Deforestation': {'color': '#f4a259', 'hatch': '//'},
    'Reafforestation': {'color': '#6dbf6b', 'hatch': '///'},
}

fig, ax = plt.subplots(figsize=(11, 6))

for scenario_index, (scenario_column, scenario_label) in enumerate(scenario_columns):
    offsets = x + (scenario_index - 1) * bar_width
    ax.bar(
        offsets,
        sector_chart_musd[scenario_column].to_numpy(),
        width=bar_width,
        label=scenario_label,
        color=scenario_styles[scenario_label]['color'],
        edgecolor='#3a3a3a',
        linewidth=0.8,
        hatch=scenario_styles[scenario_label]['hatch'],
    )

if len(sector_chart_musd.index) >= 2:
    ax.axvline(len(sector_chart_musd.index) - 1.5, linestyle=':', color='#808080', linewidth=1.1)

x_tick_labels = [sector_label.capitalize() if sector_label != 'TOTAL' else 'TOTAL' for sector_label in sector_chart_musd.index]
ax.set_xticks(x)
ax.set_xticklabels(x_tick_labels)
ax.set_ylabel('EAD (USD million)')
ax.set_title('Landslide Combined-Class EAD by Sector and Scenario')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=True, ncol=3, loc='upper left')

plt.tight_layout()

scenario_chart_file = output_damage_estimates / 'landslide_source_and_runout_sector_total_ead_usd_by_scenario_grouped_combined_class.png'
fig.savefig(scenario_chart_file, dpi=300, bbox_inches='tight')
print('Saved chart:', scenario_chart_file)
plt.show()

display(sector_chart.reset_index().rename(columns={'index': 'Sector'}))


In [ ]:
# Stacked bar chart: baseline with reafforestation and deforestation changes (USD million)
stack_sector_order = ['buildings', 'transport', 'water', 'energy']

stack_table = (
    asset_ead
    .groupby('Sector', as_index=False)[['EAD_Baseline_USD', 'EAD_Reafforestation_USD', 'EAD_Deforestation_USD']]
    .sum()
    .set_index('Sector')
)

for sector_name in stack_sector_order:
    if sector_name not in stack_table.index:
        stack_table.loc[sector_name, ['EAD_Baseline_USD', 'EAD_Reafforestation_USD', 'EAD_Deforestation_USD']] = 0.0

stack_table = stack_table.loc[stack_sector_order]
stack_table.loc['TOTAL', 'EAD_Baseline_USD'] = float(stack_table['EAD_Baseline_USD'].sum())
stack_table.loc['TOTAL', 'EAD_Reafforestation_USD'] = float(stack_table['EAD_Reafforestation_USD'].sum())
stack_table.loc['TOTAL', 'EAD_Deforestation_USD'] = float(stack_table['EAD_Deforestation_USD'].sum())

stack_table['Reafforestation_Change_vs_Baseline_USD'] = stack_table['EAD_Baseline_USD'] - stack_table['EAD_Reafforestation_USD']
stack_table['Deforestation_Change_vs_Baseline_USD'] = stack_table['EAD_Deforestation_USD'] - stack_table['EAD_Baseline_USD']

stack_table['Baseline_USD_million'] = stack_table['EAD_Baseline_USD'] / 1_000_000.0
stack_table['Reafforestation_Change_USD_million'] = stack_table['Reafforestation_Change_vs_Baseline_USD'] / 1_000_000.0
stack_table['Deforestation_Change_USD_million'] = stack_table['Deforestation_Change_vs_Baseline_USD'] / 1_000_000.0

stack_table['Reafforestation_Change_Share_Pct'] = np.where(
    stack_table['EAD_Baseline_USD'] > 0,
    100.0 * stack_table['Reafforestation_Change_vs_Baseline_USD'] / stack_table['EAD_Baseline_USD'],
    np.nan,
)
stack_table['Deforestation_Change_Share_Pct'] = np.where(
    stack_table['EAD_Baseline_USD'] > 0,
    100.0 * stack_table['Deforestation_Change_vs_Baseline_USD'] / stack_table['EAD_Baseline_USD'],
    np.nan,
)

x = np.arange(len(stack_table.index))
baseline_values = stack_table['Baseline_USD_million'].to_numpy()
reafforestation_change_values = stack_table['Reafforestation_Change_USD_million'].to_numpy()
deforestation_change_values = stack_table['Deforestation_Change_USD_million'].to_numpy()

positive_reafforestation_change = np.clip(reafforestation_change_values, 0, None)
negative_reafforestation_change = np.clip(reafforestation_change_values, None, 0)
positive_deforestation_change = np.clip(deforestation_change_values, 0, None)
negative_deforestation_change = np.clip(deforestation_change_values, None, 0)

fig, ax = plt.subplots(figsize=(11.8, 6.8))

ax.bar(
    x,
    baseline_values,
    color='#c0c0c0',
    edgecolor='#4d4d4d',
    linewidth=0.8,
    label='Baseline EAD',
)

# Stack positive changes above baseline (green then orange).
ax.bar(
    x,
    positive_reafforestation_change,
    bottom=baseline_values,
    color='#d9f0d3',
    edgecolor='#2b8c3e',
    linewidth=0.8,
    hatch='///',
    label='Reafforestation compared with baseline (Baseline - Reafforestation)',
)
ax.bar(
    x,
    positive_deforestation_change,
    bottom=baseline_values + positive_reafforestation_change,
    color='#fdd49e',
    edgecolor='#d95f0e',
    linewidth=0.8,
    hatch='xx',
    label='Deforestation compared with baseline (Deforestation - Baseline)',
)

# Stack negative changes below baseline if present.
if (negative_reafforestation_change < 0).any():
    ax.bar(
        x,
        negative_reafforestation_change,
        bottom=baseline_values,
        color='#c7e9c0',
        edgecolor='#238b45',
        linewidth=0.8,
        hatch='\\',
        label='Reafforestation higher than baseline',
    )
if (negative_deforestation_change < 0).any():
    ax.bar(
        x,
        negative_deforestation_change,
        bottom=baseline_values + negative_reafforestation_change,
        color='#deebf7',
        edgecolor='#3182bd',
        linewidth=0.8,
        hatch='..',
        label='Deforestation lower than baseline',
    )

if len(stack_table.index) >= 2:
    ax.axvline(len(stack_table.index) - 1.5, linestyle=':', color='#808080', linewidth=1.1)

x_tick_labels = [sector_label.capitalize() if sector_label != 'TOTAL' else 'TOTAL' for sector_label in stack_table.index]
ax.set_xticks(x)
ax.set_xticklabels(x_tick_labels)
ax.set_ylabel('EAD (USD million)')
ax.set_title('Scenario Comparison Against Baseline EAD')
ax.grid(axis='y', alpha=0.25)

legend_handles, legend_labels = ax.get_legend_handles_labels()
seen_labels = set()
unique_legend_handles = []
unique_legend_labels = []
for legend_handle, legend_label in zip(legend_handles, legend_labels):
    if legend_label not in seen_labels and legend_label:
        seen_labels.add(legend_label)
        unique_legend_handles.append(legend_handle)
        unique_legend_labels.append(legend_label)
ax.legend(unique_legend_handles, unique_legend_labels, frameon=True, loc='upper left')

for x_position, baseline_value, reafforestation_pct, deforestation_pct in zip(
    x,
    baseline_values,
    stack_table['Reafforestation_Change_Share_Pct'].to_numpy(),
    stack_table['Deforestation_Change_Share_Pct'].to_numpy(),
):
    if np.isfinite(reafforestation_pct):
        ax.text(x_position, baseline_value, f"R {reafforestation_pct:+.1f}%", ha='center', va='bottom', fontsize=8, color='#238b45')
    if np.isfinite(deforestation_pct):
        ax.text(x_position, baseline_value * 0.985, f"D {deforestation_pct:+.1f}%", ha='center', va='top', fontsize=8, color='#d95f0e')

plt.tight_layout()

stacked_chart_file = output_damage_estimates / 'landslide_baseline_with_scenario_changes_stacked_combined_class.png'
fig.savefig(stacked_chart_file, dpi=300, bbox_inches='tight')
print('Saved chart:', stacked_chart_file)
plt.show()

display(
    stack_table.reset_index().rename(columns={'index': 'Sector'})[
        [
            'Sector',
            'EAD_Baseline_USD', 'EAD_Reafforestation_USD', 'EAD_Deforestation_USD',
            'Reafforestation_Change_vs_Baseline_USD', 'Deforestation_Change_vs_Baseline_USD',
            'Reafforestation_Change_Share_Pct', 'Deforestation_Change_Share_Pct'
        ]
    ]
)


In [ ]:
# Alternative chart gallery for scenario comparison
alt_sector_order = ['buildings', 'transport', 'water', 'energy']

alternative_chart_table = (
    asset_ead
    .groupby('Sector', as_index=False)[['EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD']]
    .sum()
    .set_index('Sector')
)

for sector_name in alt_sector_order:
    if sector_name not in alternative_chart_table.index:
        alternative_chart_table.loc[sector_name, ['EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD']] = 0.0

alternative_chart_table = alternative_chart_table.loc[alt_sector_order]
alternative_chart_table.loc['TOTAL', 'EAD_Baseline_USD'] = float(alternative_chart_table['EAD_Baseline_USD'].sum())
alternative_chart_table.loc['TOTAL', 'EAD_Deforestation_USD'] = float(alternative_chart_table['EAD_Deforestation_USD'].sum())
alternative_chart_table.loc['TOTAL', 'EAD_Reafforestation_USD'] = float(alternative_chart_table['EAD_Reafforestation_USD'].sum())

alternative_chart_table['baseline_musd'] = alternative_chart_table['EAD_Baseline_USD'] / 1_000_000.0
alternative_chart_table['deforestation_musd'] = alternative_chart_table['EAD_Deforestation_USD'] / 1_000_000.0
alternative_chart_table['reafforestation_musd'] = alternative_chart_table['EAD_Reafforestation_USD'] / 1_000_000.0
alternative_chart_table['deforestation_change_musd'] = alternative_chart_table['deforestation_musd'] - alternative_chart_table['baseline_musd']
alternative_chart_table['reafforestation_change_musd'] = alternative_chart_table['reafforestation_musd'] - alternative_chart_table['baseline_musd']

alternative_chart_table['deforestation_index'] = np.where(alternative_chart_table['EAD_Baseline_USD'] > 0, 100.0 * alternative_chart_table['EAD_Deforestation_USD'] / alternative_chart_table['EAD_Baseline_USD'], np.nan)
alternative_chart_table['reafforestation_index'] = np.where(alternative_chart_table['EAD_Baseline_USD'] > 0, 100.0 * alternative_chart_table['EAD_Reafforestation_USD'] / alternative_chart_table['EAD_Baseline_USD'], np.nan)

sector_axis_labels = [sector_label.capitalize() if sector_label != 'TOTAL' else 'TOTAL' for sector_label in alternative_chart_table.index]
y = np.arange(len(alternative_chart_table.index))

# Alt 1: dumbbell plot (absolute EAD, USD million)
fig, ax = plt.subplots(figsize=(10.5, 6.5))
for row_position, scenario_row in enumerate(alternative_chart_table.itertuples()):
    scenario_values = [scenario_row.reafforestation_musd, scenario_row.baseline_musd, scenario_row.deforestation_musd]
    ax.plot([min(scenario_values), max(scenario_values)], [row_position, row_position], color='#c7c7c7', linewidth=2, zorder=1)

ax.scatter(alternative_chart_table['reafforestation_musd'], y, color='#2b8c3e', s=55, label='Reafforestation', zorder=3)
ax.scatter(alternative_chart_table['baseline_musd'], y, color='#7f7f7f', s=55, label='Baseline', zorder=3)
ax.scatter(alternative_chart_table['deforestation_musd'], y, color='#d95f0e', s=55, label='Deforestation', zorder=3)

ax.set_yticks(y)
ax.set_yticklabels(sector_axis_labels)
ax.set_xlabel('EAD (USD million)')
ax.set_title('Alternative 1: Dumbbell Comparison by Sector')
ax.grid(axis='x', alpha=0.25)
ax.legend(frameon=True, loc='lower right')
plt.tight_layout()

alt1_file = output_damage_estimates / 'landslide_source_and_runout_alt_01_dumbbell_scenarios_combined_class.png'
fig.savefig(alt1_file, dpi=300, bbox_inches='tight')
print('Saved chart:', alt1_file)
plt.show()

# Alt 2: diverging bars of change vs baseline (USD million)
fig, ax = plt.subplots(figsize=(10.5, 6.5))
change_bar_width = 0.36
ax.barh(y - change_bar_width/2, alternative_chart_table['reafforestation_change_musd'], height=change_bar_width, color='#2b8c3e', alpha=0.85, label='Reafforestation - Baseline')
ax.barh(y + change_bar_width/2, alternative_chart_table['deforestation_change_musd'], height=change_bar_width, color='#d95f0e', alpha=0.85, label='Deforestation - Baseline')
ax.axvline(0, color='#4d4d4d', linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(sector_axis_labels)
ax.set_xlabel('Change vs baseline (USD million)')
ax.set_title('Alternative 2: Diverging Change from Baseline')
ax.grid(axis='x', alpha=0.25)
ax.legend(frameon=True, loc='lower right')
plt.tight_layout()

alt2_file = output_damage_estimates / 'landslide_source_and_runout_alt_02_diverging_change_vs_baseline_combined_class.png'
fig.savefig(alt2_file, dpi=300, bbox_inches='tight')
print('Saved chart:', alt2_file)
plt.show()

# Alt 3: baseline-index chart (Baseline = 100)
fig, ax = plt.subplots(figsize=(10.8, 6.5))
x = np.arange(len(alternative_chart_table.index))
baseline_index_bar_width = 0.25
ax.bar(x - baseline_index_bar_width, np.full(len(x), 100.0), width=baseline_index_bar_width, color='#bdbdbd', edgecolor='#4d4d4d', linewidth=0.8, label='Baseline (100)')
ax.bar(x, alternative_chart_table['reafforestation_index'], width=baseline_index_bar_width, color='#74c476', edgecolor='#238b45', linewidth=0.8, label='Reafforestation index')
ax.bar(x + baseline_index_bar_width, alternative_chart_table['deforestation_index'], width=baseline_index_bar_width, color='#fdae6b', edgecolor='#d95f0e', linewidth=0.8, label='Deforestation index')
ax.axhline(100, color='#4d4d4d', linewidth=1, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(sector_axis_labels)
ax.set_ylabel('Index (Baseline = 100)')
ax.set_title('Alternative 3: Relative Scenario Index by Sector')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=True, loc='upper left')
plt.tight_layout()

alt3_file = output_damage_estimates / 'landslide_source_and_runout_alt_03_baseline_index_comparison_combined_class.png'
fig.savefig(alt3_file, dpi=300, bbox_inches='tight')
print('Saved chart:', alt3_file)
plt.show()

# Alt 4: small multiples (absolute EAD, USD million)
fig, axes = plt.subplots(1, len(alternative_chart_table.index), figsize=(3.2 * len(alternative_chart_table.index), 5.6), sharey=True)
if len(alternative_chart_table.index) == 1:
    axes = [axes]

scenario_names = ['Baseline', 'Reafforestation', 'Deforestation']
scenario_colors = ['#7f7f7f', '#2b8c3e', '#d95f0e']

for sector_index, (sector_label, scenario_row) in enumerate(zip(sector_axis_labels, alternative_chart_table.itertuples())):
    scenario_values = [scenario_row.baseline_musd, scenario_row.reafforestation_musd, scenario_row.deforestation_musd]
    ax = axes[sector_index]
    ax.bar(scenario_names, scenario_values, color=scenario_colors, edgecolor='#3a3a3a', linewidth=0.6)
    ax.set_title(sector_label)
    ax.tick_params(axis='x', rotation=90)
    ax.grid(axis='y', alpha=0.22)

axes[0].set_ylabel('EAD (USD million)')
fig.suptitle('Alternative 4: Small Multiples by Sector', y=1.02)
plt.tight_layout()

alt4_file = output_damage_estimates / 'landslide_source_and_runout_alt_04_small_multiples_scenarios_combined_class.png'
fig.savefig(alt4_file, dpi=300, bbox_inches='tight')
print('Saved chart:', alt4_file)
plt.show()

display(
    alternative_chart_table.reset_index().rename(columns={'index': 'Sector'})[
        [
            'Sector',
            'EAD_Baseline_USD', 'EAD_Reafforestation_USD', 'EAD_Deforestation_USD',
            'reafforestation_change_musd', 'deforestation_change_musd',
            'reafforestation_index', 'deforestation_index'
        ]
    ]
)

In [ ]:
# Mapping helpers
jamaica_boundary = gpd.read_file(jamaica_boundary_path).to_crs('EPSG:3448')
map_out_dir = output_damage_estimates / 'maps_landslide_source_and_runout_avoided_ead_combined_class'
map_out_dir.mkdir(parents=True, exist_ok=True)

diverging_damage_colormap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)

def choose_unit(abs_max_usd):
    if abs_max_usd >= 1e6:
        return 1e6, 'USD millions'
    if abs_max_usd >= 1e3:
        return 1e3, 'USD thousands'
    return 1.0, 'USD'


def add_north_arrow(ax):
    ax.annotate(
        'N',
        xy=(0.94, 0.90),
        xytext=(0.94, 0.78),
        xycoords='axes fraction',
        textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black', width=3, headwidth=10, headlength=12),
        ha='center',
        va='center',
        fontsize=12,
        fontweight='bold',
        zorder=20,
    )


def add_scale_bar(ax, length_km=25):
    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()
    x_range = x_max - x_min
    y_range = y_max - y_min
    length_m = length_km * 1000.0
    x_start = x_min + 0.08 * x_range
    y_start = y_min + 0.07 * y_range
    tick_height = 0.01 * y_range

    ax.plot([x_start, x_start + length_m], [y_start, y_start], color='black', linewidth=3, solid_capstyle='butt', zorder=20)
    ax.plot([x_start, x_start], [y_start - tick_height, y_start + tick_height], color='black', linewidth=2, zorder=20)
    ax.plot([x_start + length_m, x_start + length_m], [y_start - tick_height, y_start + tick_height], color='black', linewidth=2, zorder=20)
    ax.text(
        x_start + (length_m / 2.0),
        y_start + (2.2 * tick_height),
        f'{length_km:g} km',
        ha='center',
        va='bottom',
        fontsize=9,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=1.5),
        zorder=20,
    )


def apply_jamaica_map_formatting(ax):
    add_north_arrow(ax)
    add_scale_bar(ax)
    ax.set_axis_off()



def plot_metric_map(gdf, value_col, title, output_png, display_quantile=0.995):
    values_usd = pd.to_numeric(gdf[value_col], errors='coerce').fillna(0.0)
    abs_values = values_usd.abs()
    cap_usd = float(abs_values.quantile(display_quantile))
    if cap_usd <= 0:
        cap_usd = float(abs_values.max()) if float(abs_values.max()) > 0 else 1.0

    unit_factor, unit_label = choose_unit(cap_usd)
    cap = cap_usd / unit_factor

    plot_gdf = gdf.copy()
    plot_gdf['_plot_val'] = (values_usd / unit_factor).clip(-cap, cap)

    norm = TwoSlopeNorm(vmin=-cap, vcenter=0.0, vmax=cap)

    fig, ax = plt.subplots(figsize=(11, 10))
    ax.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

    geom_type = plot_gdf.geometry.geom_type.astype(str)
    polygons = plot_gdf[geom_type.str.contains('Polygon', na=False)]
    lines = plot_gdf[geom_type.str.contains('LineString', na=False)]
    points = plot_gdf[geom_type.str.contains('Point', na=False)]

    if not polygons.empty:
        polygons.plot(ax=ax, column='_plot_val', cmap=diverging_damage_colormap, norm=norm, linewidth=0.12, edgecolor='none', alpha=0.9, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column='_plot_val', cmap=diverging_damage_colormap, norm=norm, linewidth=0.9, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column='_plot_val', cmap=diverging_damage_colormap, norm=norm, markersize=14, alpha=0.95, zorder=4)

    scalar_mappable = ScalarMappable(norm=norm, cmap=diverging_damage_colormap)
    scalar_mappable.set_array([])
    colorbar = fig.colorbar(scalar_mappable, ax=ax, fraction=0.036, pad=0.02)
    colorbar.set_label(f'{value_col} ({unit_label}), clipped at q={display_quantile:.3f}', rotation=90)

    ax.set_title(title, fontsize=13)
    apply_jamaica_map_formatting(ax)
    plt.tight_layout()

    fig.savefig(output_png, dpi=300, bbox_inches='tight')
    print('Saved:', output_png)
    plt.show()


In [ ]:
# Map 1: Avoided EAD from reafforestation (Baseline - Reafforestation)
plot_metric_map(
    landslide_ead_map_layers,
    value_col='Avoided_EAD_Reafforestation_USD',
    title='Avoided EAD from Reafforestation (Baseline - Reafforestation)',
    output_png=map_out_dir / 'avoided_ead_reafforestation_all_sectors_combined_class.png',
)


In [ ]:
# Map 2: Avoided EAD from protecting forests (Deforestation - Baseline)
plot_metric_map(
    landslide_ead_map_layers,
    value_col='Avoided_EAD_Protection_USD',
    title='Avoided EAD from Protecting Forests (Deforestation - Baseline)',
    output_png=map_out_dir / 'avoided_ead_protection_all_sectors_combined_class.png',
)


In [ ]:
# Optional: sector-specific maps for each avoided EAD metric
sector_order = ['buildings', 'energy', 'transport', 'water']
metrics = [
    ('Avoided_EAD_Reafforestation_USD', 'reafforestation'),
    ('Avoided_EAD_Protection_USD', 'protection'),
]

for metric_col, metric_label in metrics:
    for sector_name in sector_order:
        sector_gdf = landslide_ead_map_layers[landslide_ead_map_layers['Sector'] == sector_name].copy()
        if sector_gdf.empty:
            continue

        out_png = map_out_dir / f'avoided_ead_{metric_label}_{sector_name}_combined_class.png'
        plot_metric_map(
            sector_gdf,
            value_col=metric_col,
            title=f'{metric_col} - {sector_name.capitalize()}',
            output_png=out_png,
        )


In [ ]:
# Forest-associated increase maps (where forested scenarios have higher damages)
# Positive values represent locations where damages are higher with forests.
# 1) Protection comparison (Deforestation - Baseline):
#    if negative, then Baseline > Deforestation => forests associated with higher damages.
# 2) Reafforestation comparison (Baseline - Reafforestation):
#    if negative, then Reafforestation > Baseline => reafforestation associated with higher damages.

landslide_ead_map_layers['Forest_Associated_Increase_Protection_USD'] = (
    -pd.to_numeric(landslide_ead_map_layers['Avoided_EAD_Protection_USD'], errors='coerce').fillna(0.0)
).clip(lower=0.0)

landslide_ead_map_layers['Forest_Associated_Increase_Reafforestation_USD'] = (
    -pd.to_numeric(landslide_ead_map_layers['Avoided_EAD_Reafforestation_USD'], errors='coerce').fillna(0.0)
).clip(lower=0.0)


def plot_positive_only_map(gdf, value_col, title, output_png, display_quantile=0.995):
    values_usd = pd.to_numeric(gdf[value_col], errors='coerce').fillna(0.0)
    positive_mask = values_usd > 0

    if positive_mask.sum() == 0:
        print(f'No features with positive {value_col}; no map generated: {output_png.name}')
        return False

    plot_gdf = gdf.loc[positive_mask].copy()
    values_usd = values_usd.loc[positive_mask]

    cap_usd = float(values_usd.quantile(display_quantile))
    if cap_usd <= 0:
        cap_usd = float(values_usd.max()) if float(values_usd.max()) > 0 else 1.0

    unit_factor, unit_label = choose_unit(cap_usd)
    cap = cap_usd / unit_factor

    plot_gdf['_plot_val'] = (values_usd / unit_factor).clip(0, cap)
    norm = Normalize(vmin=0, vmax=cap)

    increase_diverging_damage_colormap = LinearSegmentedColormap.from_list('white_red', ['#ffffff', '#cb181d'], N=256)

    fig, ax = plt.subplots(figsize=(11, 10))
    ax.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

    geom_type = plot_gdf.geometry.geom_type.astype(str)
    polygons = plot_gdf[geom_type.str.contains('Polygon', na=False)]
    lines = plot_gdf[geom_type.str.contains('LineString', na=False)]
    points = plot_gdf[geom_type.str.contains('Point', na=False)]

    if not polygons.empty:
        polygons.plot(ax=ax, column='_plot_val', cmap=increase_diverging_damage_colormap, norm=norm, linewidth=0.12, edgecolor='none', alpha=0.9, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column='_plot_val', cmap=increase_diverging_damage_colormap, norm=norm, linewidth=0.9, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column='_plot_val', cmap=increase_diverging_damage_colormap, norm=norm, markersize=14, alpha=0.95, zorder=4)

    scalar_mappable = ScalarMappable(norm=norm, cmap=increase_diverging_damage_colormap)
    scalar_mappable.set_array([])
    colorbar = fig.colorbar(scalar_mappable, ax=ax, fraction=0.036, pad=0.02)
    colorbar.set_label(f'{value_col} ({unit_label}), clipped at q={display_quantile:.3f}', rotation=90)

    ax.set_title(title, fontsize=13)
    apply_jamaica_map_formatting(ax)
    plt.tight_layout()

    fig.savefig(output_png, dpi=300, bbox_inches='tight')
    print('Saved:', output_png)
    plt.show()
    return True


# Save asset-level table for colleague review
forest_increase_assets = landslide_ead_map_layers.loc[
    (landslide_ead_map_layers['Forest_Associated_Increase_Protection_USD'] > 0)
    | (landslide_ead_map_layers['Forest_Associated_Increase_Reafforestation_USD'] > 0),
    [
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
        'EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD',
        'Forest_Associated_Increase_Protection_USD',
        'Forest_Associated_Increase_Reafforestation_USD',
        'geometry'
    ]
].copy()

forest_increase_asset_csv = output_damage_estimates / 'landslide_source_and_runout_forest_associated_damage_increase_asset_level_usd_combined_class.csv'
forest_increase_assets.drop(columns=['geometry']).to_csv(forest_increase_asset_csv, index=False)
print('Saved:', forest_increase_asset_csv)

# Totals and sector totals
forest_increase_totals = pd.DataFrame([{
    'Forest_Associated_Increase_Protection_USD': float(landslide_ead_map_layers['Forest_Associated_Increase_Protection_USD'].sum()),
    'Forest_Associated_Increase_Reafforestation_USD': float(landslide_ead_map_layers['Forest_Associated_Increase_Reafforestation_USD'].sum()),
    'Protection_Increase_Asset_Count': int((landslide_ead_map_layers['Forest_Associated_Increase_Protection_USD'] > 0).sum()),
    'Reafforestation_Increase_Asset_Count': int((landslide_ead_map_layers['Forest_Associated_Increase_Reafforestation_USD'] > 0).sum()),
}])
forest_increase_totals['Forest_Associated_Increase_Protection_Readable'] = forest_increase_totals['Forest_Associated_Increase_Protection_USD'].apply(format_usd_readable)
forest_increase_totals['Forest_Associated_Increase_Reafforestation_Readable'] = forest_increase_totals['Forest_Associated_Increase_Reafforestation_USD'].apply(format_usd_readable)

forest_increase_totals_csv = output_damage_estimates / 'landslide_source_and_runout_forest_associated_damage_increase_totals_usd_combined_class.csv'
forest_increase_totals.to_csv(forest_increase_totals_csv, index=False)
print('Saved:', forest_increase_totals_csv)

forest_increase_sector = (
    landslide_ead_map_layers
    .groupby('Sector', as_index=False)[
        ['Forest_Associated_Increase_Protection_USD', 'Forest_Associated_Increase_Reafforestation_USD']
    ]
    .sum()
)
forest_increase_sector['Forest_Associated_Increase_Protection_Readable'] = forest_increase_sector['Forest_Associated_Increase_Protection_USD'].apply(format_usd_readable)
forest_increase_sector['Forest_Associated_Increase_Reafforestation_Readable'] = forest_increase_sector['Forest_Associated_Increase_Reafforestation_USD'].apply(format_usd_readable)

forest_increase_sector_csv = output_damage_estimates / 'landslide_source_and_runout_forest_associated_damage_increase_sector_totals_usd_combined_class.csv'
forest_increase_sector.to_csv(forest_increase_sector_csv, index=False)
print('Saved:', forest_increase_sector_csv)

# Maps (positive-only)
forest_increase_map_protection = map_out_dir / 'forest_associated_damage_increase_protection_only_all_sectors_combined_class.png'
forest_increase_map_reafforestation = map_out_dir / 'forest_associated_damage_increase_reafforestation_only_all_sectors_combined_class.png'

plot_positive_only_map(
    landslide_ead_map_layers,
    value_col='Forest_Associated_Increase_Protection_USD',
    title='Damage Increase Where Baseline > Deforestation (Forests-associated increase)',
    output_png=forest_increase_map_protection,
)

has_reafforestation_increase = plot_positive_only_map(
    landslide_ead_map_layers,
    value_col='Forest_Associated_Increase_Reafforestation_USD',
    title='Damage Increase Where Reafforestation > Baseline (Reafforestation-associated increase)',
    output_png=forest_increase_map_reafforestation,
)

if not has_reafforestation_increase:
    no_map_note = output_damage_estimates / 'forest_associated_damage_increase_reafforestation_none_combined_class.txt'
    no_map_note.write_text('No assets with positive Forest_Associated_Increase_Reafforestation_USD in this scenario run.\n', encoding='utf-8')
    print('Saved:', no_map_note)

print('Forest-associated increase totals:')
display(forest_increase_totals)
print('By sector:')
display(forest_increase_sector.sort_values('Sector'))


# High-contrast highlight maps (binary: any increase shown in red)
def plot_binary_increase_map(gdf, value_col, title, output_png):
    values = pd.to_numeric(gdf[value_col], errors='coerce').fillna(0.0)
    inc_mask = values > 0

    if inc_mask.sum() == 0:
        print(f'No positive {value_col}; no highlight map generated: {output_png.name}')
        return False

    inc = gdf.loc[inc_mask].copy()
    total_increase = float(values.loc[inc_mask].sum())

    fig, ax = plt.subplots(figsize=(11, 10))
    ax.set_facecolor('#ffffff')

    # Jamaica outline
    jamaica_boundary.boundary.plot(ax=ax, color='#9e9e9e', linewidth=0.6, zorder=1)

    # Context: all mapped assets in very light grey
    gt_all = gdf.geometry.geom_type.astype(str)
    all_polygons = gdf[gt_all.str.contains('Polygon', na=False)]
    all_lines = gdf[gt_all.str.contains('LineString', na=False)]
    all_points = gdf[gt_all.str.contains('Point', na=False)]

    if not all_polygons.empty:
        all_polygons.plot(ax=ax, color='#efefef', edgecolor='none', alpha=0.55, zorder=2)
    if not all_lines.empty:
        all_lines.plot(ax=ax, color='#cfcfcf', linewidth=0.35, alpha=0.8, zorder=2)
    if not all_points.empty:
        all_points.plot(ax=ax, color='#d9d9d9', markersize=8, alpha=0.8, zorder=2)

    # Highlight increases in strong red
    gt_inc = inc.geometry.geom_type.astype(str)
    inc_polygons = inc[gt_inc.str.contains('Polygon', na=False)]
    inc_lines = inc[gt_inc.str.contains('LineString', na=False)]
    inc_points = inc[gt_inc.str.contains('Point', na=False)]

    if not inc_polygons.empty:
        inc_polygons.plot(ax=ax, color='#ff2d2d', edgecolor='#990000', linewidth=0.20, alpha=0.90, zorder=3)
    if not inc_lines.empty:
        inc_lines.plot(ax=ax, color='#ff0000', linewidth=1.6, alpha=0.95, zorder=4)
    if not inc_points.empty:
        inc_points.plot(ax=ax, color='#ff0000', markersize=26, alpha=0.95, zorder=5)

    ax.set_title(title, fontsize=13)
    ax.text(
        0.01, 0.01,
        f"Highlighted assets: {int(inc_mask.sum()):,} | Total increase: {format_usd_readable(total_increase)}",
        transform=ax.transAxes,
        ha='left', va='bottom',
        fontsize=10,
        bbox=dict(facecolor='white', edgecolor='#bbbbbb', alpha=0.9)
    )
    apply_jamaica_map_formatting(ax)
    plt.tight_layout()

    fig.savefig(output_png, dpi=300, bbox_inches='tight')
    print('Saved:', output_png)
    plt.show()
    return True


forest_increase_map_protection_highlight = map_out_dir / 'forest_associated_damage_increase_protection_highlight_red_all_sectors_combined_class.png'
forest_increase_map_reafforestation_highlight = map_out_dir / 'forest_associated_damage_increase_reafforestation_highlight_red_all_sectors_combined_class.png'

plot_binary_increase_map(
    landslide_ead_map_layers,
    value_col='Forest_Associated_Increase_Protection_USD',
    title='Any Increase with Forests Highlighted in Red (Baseline > Deforestation)',
    output_png=forest_increase_map_protection_highlight,
)

has_reafforestation_highlight = plot_binary_increase_map(
    landslide_ead_map_layers,
    value_col='Forest_Associated_Increase_Reafforestation_USD',
    title='Any Reafforestation-Associated Increase Highlighted in Red (Reafforestation > Baseline)',
    output_png=forest_increase_map_reafforestation_highlight,
)

if not has_reafforestation_highlight:
    no_map_note2 = output_damage_estimates / 'forest_associated_damage_increase_reafforestation_highlight_none_combined_class.txt'
    no_map_note2.write_text('No reafforestation-associated increase locations to highlight in red.\n', encoding='utf-8')
    print('Saved:', no_map_note2)


In [ ]:
# Intersect forest-associated increase areas with land-use map
landuse_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_landcover.gpkg'
landuse_layer = '2013_landuse_landcover'

if not landuse_path.exists():
    raise FileNotFoundError(f'Missing land-use dataset: {landuse_path}')

landuse = gpd.read_file(landuse_path, layer=landuse_layer)
if landuse.crs is not None and str(landuse.crs).upper() != 'EPSG:3448':
    landuse = landuse.to_crs('EPSG:3448')

if 'Classify' not in landuse.columns:
    raise KeyError("Land-use layer missing 'Classify' column")
if 'LU_CODE' not in landuse.columns:
    landuse['LU_CODE'] = pd.NA

landuse = landuse[['Classify', 'LU_CODE', 'geometry']].copy()
landuse = landuse[landuse.geometry.notnull()].copy()
landuse['geometry'] = landuse.geometry.buffer(0)
landuse = landuse[~landuse.geometry.is_empty].copy()


def allocate_increase_to_landuse(increase_col, prefix):
    source = landslide_ead_map_layers.loc[
        landslide_ead_map_layers[increase_col] > 0,
        ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', increase_col, 'geometry']
    ].copy()

    # Save empty outputs if none.
    out_csv = output_damage_estimates / f'landslide_{prefix}_landuse_class_summary_usd_combined_class.csv'
    out_gpkg = output_damage_estimates / f'landslide_{prefix}_landuse_intersection_combined_class.gpkg'

    if source.empty:
        empty_landuse_summary = pd.DataFrame(columns=[
            'Classify', 'LU_CODE',
            'Allocated_Increase_USD',
            'Affected_Asset_Count',
            'Intersection_Length_km',
            'Intersection_Area_sqkm',
            'Affected_Point_Count'
        ])
        empty_landuse_summary.to_csv(out_csv, index=False)
        print(f'No positive {increase_col}; wrote empty summary:', out_csv)
        return empty_landuse_summary

    source = source[source.geometry.notnull()].copy()
    source['geometry'] = source.geometry.buffer(0)
    source = source[~source.geometry.is_empty].copy()

    geom_type = source.geometry.geom_type.astype(str)
    points = source[geom_type.str.contains('Point', na=False)].copy()
    lines = source[geom_type.str.contains('LineString', na=False)].copy()
    polygons = source[geom_type.str.contains('Polygon', na=False)].copy()

    pieces = []

    # Points: spatial join with land-use polygons.
    if not points.empty:
        point_landuse_join = gpd.sjoin(
            points,
            landuse[['Classify', 'LU_CODE', 'geometry']],
            how='left',
            predicate='within'
        )
        point_landuse_join['Allocated_Increase_USD'] = point_landuse_join[increase_col]
        point_landuse_join['Intersection_Length_km'] = 0.0
        point_landuse_join['Intersection_Area_sqkm'] = 0.0
        point_landuse_join['Affected_Point_Count'] = 1
        point_landuse_join['Asset_Geom_Type'] = 'Point'
        pieces.append(point_landuse_join[['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Classify', 'LU_CODE',
                             'Allocated_Increase_USD', 'Intersection_Length_km', 'Intersection_Area_sqkm',
                             'Affected_Point_Count', 'Asset_Geom_Type', 'geometry']])

    # Lines: intersect with land-use polygons and allocate by length share.
    if not lines.empty:
        lines = lines.copy()
        lines['orig_measure'] = lines.geometry.length.replace(0, pd.NA)
        line_landuse_intersections = gpd.overlay(
            lines,
            landuse[['Classify', 'LU_CODE', 'geometry']],
            how='intersection',
            keep_geom_type=False,
        )
        if not line_landuse_intersections.empty:
            line_landuse_intersections = line_landuse_intersections[line_landuse_intersections.geometry.notnull()].copy()
            line_landuse_intersections = line_landuse_intersections[~line_landuse_intersections.geometry.is_empty].copy()
            line_landuse_intersections['piece_measure'] = line_landuse_intersections.geometry.length
            line_landuse_intersections['weight'] = (line_landuse_intersections['piece_measure'] / line_landuse_intersections['orig_measure']).fillna(0.0)
            line_landuse_intersections['Allocated_Increase_USD'] = line_landuse_intersections[increase_col] * line_landuse_intersections['weight']
            line_landuse_intersections['Intersection_Length_km'] = line_landuse_intersections['piece_measure'] / 1000.0
            line_landuse_intersections['Intersection_Area_sqkm'] = 0.0
            line_landuse_intersections['Affected_Point_Count'] = 0
            line_landuse_intersections['Asset_Geom_Type'] = 'LineString'
            pieces.append(line_landuse_intersections[['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Classify', 'LU_CODE',
                                   'Allocated_Increase_USD', 'Intersection_Length_km', 'Intersection_Area_sqkm',
                                   'Affected_Point_Count', 'Asset_Geom_Type', 'geometry']])

    # Polygons: intersect with land-use polygons and allocate by area share.
    if not polygons.empty:
        polygons = polygons.copy()
        polygons['orig_measure'] = polygons.geometry.area.replace(0, pd.NA)
        polygon_landuse_intersections = gpd.overlay(
            polygons,
            landuse[['Classify', 'LU_CODE', 'geometry']],
            how='intersection',
            keep_geom_type=False,
        )
        if not polygon_landuse_intersections.empty:
            polygon_landuse_intersections = polygon_landuse_intersections[polygon_landuse_intersections.geometry.notnull()].copy()
            polygon_landuse_intersections = polygon_landuse_intersections[~polygon_landuse_intersections.geometry.is_empty].copy()
            polygon_landuse_intersections['piece_measure'] = polygon_landuse_intersections.geometry.area
            polygon_landuse_intersections['weight'] = (polygon_landuse_intersections['piece_measure'] / polygon_landuse_intersections['orig_measure']).fillna(0.0)
            polygon_landuse_intersections['Allocated_Increase_USD'] = polygon_landuse_intersections[increase_col] * polygon_landuse_intersections['weight']
            polygon_landuse_intersections['Intersection_Area_sqkm'] = polygon_landuse_intersections['piece_measure'] / 1_000_000.0
            polygon_landuse_intersections['Intersection_Length_km'] = 0.0
            polygon_landuse_intersections['Affected_Point_Count'] = 0
            polygon_landuse_intersections['Asset_Geom_Type'] = 'Polygon'
            pieces.append(polygon_landuse_intersections[['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Classify', 'LU_CODE',
                                   'Allocated_Increase_USD', 'Intersection_Length_km', 'Intersection_Area_sqkm',
                                   'Affected_Point_Count', 'Asset_Geom_Type', 'geometry']])

    if not pieces:
        empty_landuse_summary = pd.DataFrame(columns=[
            'Classify', 'LU_CODE',
            'Allocated_Increase_USD',
            'Affected_Asset_Count',
            'Intersection_Length_km',
            'Intersection_Area_sqkm',
            'Affected_Point_Count'
        ])
        empty_landuse_summary.to_csv(out_csv, index=False)
        print(f'No intersections created for {increase_col}; wrote empty summary:', out_csv)
        return empty_landuse_summary

    landuse_intersections = gpd.GeoDataFrame(pd.concat(pieces, ignore_index=True), geometry='geometry', crs='EPSG:3448')

    # Save map-ready intersection layer.
    landuse_intersections.to_file(out_gpkg, driver='GPKG')
    print('Saved intersection layer:', out_gpkg)

    # Summarise by land-use class.
    summary = (
        landuse_intersections.groupby(['Classify', 'LU_CODE'], dropna=False, as_index=False)
        .agg(
            Allocated_Increase_USD=('Allocated_Increase_USD', 'sum'),
            Affected_Asset_Count=('Asset_ID', lambda asset_ids: asset_ids.astype(str).nunique()),
            Intersection_Length_km=('Intersection_Length_km', 'sum'),
            Intersection_Area_sqkm=('Intersection_Area_sqkm', 'sum'),
            Affected_Point_Count=('Affected_Point_Count', 'sum'),
        )
        .sort_values('Allocated_Increase_USD', ascending=False)
        .reset_index(drop=True)
    )

    summary['Allocated_Increase_Readable'] = summary['Allocated_Increase_USD'].apply(format_usd_readable)
    summary.to_csv(out_csv, index=False)
    print('Saved summary:', out_csv)

    return summary


protection_landuse_summary = allocate_increase_to_landuse(
    'Forest_Associated_Increase_Protection_USD',
    'forest_associated_increase_protection'
)

reafforestation_landuse_summary = allocate_increase_to_landuse(
    'Forest_Associated_Increase_Reafforestation_USD',
    'forest_associated_increase_reafforestation'
)

print('Top land-use classes for protection-associated increases:')
display(protection_landuse_summary.head(20))

print('Top land-use classes for reafforestation-associated increases:')
display(reafforestation_landuse_summary.head(20))